# tth — training on a Colab GPU

Runs the three trained arms of the comparison. Runtime → Change runtime type → **T4 GPU** (free tier is enough).

| Arm | Training |
|---|---|
| A | none (bare base model, evaluated on the Mac) |
| B | shell SFT on verified Nemotron trajectories |
| C0 | chess SFT → chess GRPO (no shell data) |
| C | chess SFT → chess GRPO → the **same** shell SFT as B |
| D | table SFT → table GRPO → the same shell SFT (control) |

1. Run `scripts/pack_colab.sh` on the Mac, upload `tth_colab.zip` when the first cell asks.
2. Run all cells (~90–120 min on a T4).
3. The last cell downloads `adapters_out.zip`; unzip it in the repo root on the Mac.

No API keys are needed here: training uses only verified trajectories and local verifiers (python-chess + Stockfish).

In [ ]:
from google.colab import files
up = files.upload()  # choose tth_colab.zip
!rm -rf /content/tth && mkdir -p /content/tth && cd /content/tth && unzip -q /content/tth_colab.zip
%cd /content/tth
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install peft python-chess pydantic pyyaml
!apt-get -qq install -y stockfish > /dev/null && ln -sf /usr/games/stockfish /usr/local/bin/stockfish && stockfish quit <<< 'quit' | head -1
import os; os.environ["PYTHONPATH"] = "src:."
!wc -l data/chess/teacher_verified.jsonl data/chess/scripted_verified.jsonl data/shell/teacher_verified.jsonl

## Arm C, stage 1: chess SFT (verified Nemotron + scripted-oracle trajectories)

In [ ]:
!python -m train.sft.train_lora --data data/chess/teacher_verified.jsonl data/chess/scripted_verified.jsonl --out adapters/C_chess_sft --epochs 2 --lr 2e-4 --seed 0

## Arm C, stage 2: GRPO in the chess environment (verifier rewards only) → this checkpoint is arm C0

In [ ]:
!python -m train.grpo.grpo_chess --adapter adapters/C_chess_sft/peft --out adapters/C0_chess_grpo --rounds 2 --tasks-per-round 48 --group 4 --lr 2e-5 --seed 0

## Arm C, stage 3: shell SFT starting from the chess-trained adapter

In [ ]:
!python -m train.sft.train_lora --init-adapter adapters/C0_chess_grpo/peft --data data/shell/teacher_verified.jsonl --out adapters/C_chess_shell --epochs 2 --lr 2e-4 --seed 0

## Arm B: the same shell SFT from the base model (no chess)

In [ ]:
!python -m train.sft.train_lora --data data/shell/teacher_verified.jsonl --out adapters/B_shell --epochs 2 --lr 2e-4 --seed 0

## Arm D (control): same pipeline as C on a non-chess hidden-table domain, same episode count
If C beats D, the gain is specific to chess; if C ≈ D, any verified tool practice helps.

In [ ]:
!python -m train.sft.train_lora --data data/table/scripted_verified.jsonl --out adapters/D_table_sft --epochs 2 --lr 2e-4 --seed 0
!python -m train.grpo.grpo_chess --domain table --adapter adapters/D_table_sft/peft --out adapters/D_table_grpo --rounds 2 --tasks-per-round 48 --group 4 --lr 2e-5 --seed 0
!python -m train.sft.train_lora --init-adapter adapters/D_table_grpo/peft --data data/shell/teacher_verified.jsonl --out adapters/D_table_shell --epochs 2 --lr 2e-4 --seed 0

## Package MLX adapters + training logs and download

In [ ]:
!cd /content/tth && zip -qr adapters_out.zip adapters -x '*/peft/*' && ls -lh adapters_out.zip
files.download('/content/tth/adapters_out.zip')